# SE-ResNet18 Full + Patch V2

Notebook from-scratch pour TILDA :
- SE-ResNet18 full-image 5-fold avec TTA.
- SE-ResNet18 patch-based 5-fold avec multi-crop inference.
- Ensemble soft-probabilities full + patch.

Objectif : maximiser la generalisation sans pretrained, avec mini-batch SGD et augmentations legeres.


## 1. Setup


In [1]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = ROOT / 'data'
OUTPUT_DIR     = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'
FIGURE_DIR     = OUTPUT_DIR / 'figures'

for p in [CHECKPOINT_DIR, SUBMISSION_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Root  :', ROOT)
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('ATTENTION: CUDA indisponible. Corriger PyTorch/CUDA avant de lancer le vrai training.')


Root  : /home/onyxia/work/tilda-texture-classification
Device: cuda
GPU   : NVIDIA RTXA6000-48Q
VRAM  : 51.3 GB


## 2. Parametres

Pour un smoke test, mettre `N_SPLITS = 2` et `EPOCHS = 2`. Pour le vrai run Kaggle, garder `N_SPLITS = 5`.


In [2]:
N_SPLITS = 5
START_FOLD_FULL  = 1
START_FOLD_PATCH = 1

NUM_CLASSES = 8
NUM_WORKERS = 0

FULL_IMG_SIZE = (384, 576)   # H, W: ratio original 512:768 preserve
FULL_RESIZE   = (432, 648)

PATCH_RESIZE = (512, 768)    # garde plus de detail avant crop
PATCH_SIZE   = 384           # crop carre pour details locaux

BATCH_SIZE_FULL  = 32
BATCH_SIZE_PATCH = 32

EPOCHS_FULL  = 120
EPOCHS_PATCH = 120

LR_FULL  = 0.08
LR_PATCH = 0.08
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05
PATIENCE = 25

print('Config OK')


Config OK


## 3. Donnees

Format attendu :

```text
data/train.csv
data/train/*.tif
data/test/*.tif
```

Le notebook accepte `;` ou `,` dans `train.csv`. Les labels restent en `0..7`, comme les soumissions qui ont marche sur Kaggle.


In [3]:
def read_train_csv(path):
    try:
        df = pd.read_csv(path, sep=';')
        if len(df.columns) == 1:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if 'id' not in df.columns or 'label' not in df.columns:
        raise ValueError(f'Colonnes attendues id,label. Colonnes trouvees: {df.columns.tolist()}')
    df['id'] = df['id'].astype(int)
    df['label'] = df['label'].astype(int)
    return df


def resolve_image_path(folder, image_id):
    stem = str(int(image_id))
    for ext in ['.tif', '.tiff', '.png', '.jpg', '.jpeg']:
        p = folder / f'{stem}{ext}'
        if p.exists():
            return p
    matches = sorted(folder.glob(f'{stem}.*'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Image introuvable pour id={image_id} dans {folder}')

train_csv = DATA_DIR / 'train.csv'
train_dir = DATA_DIR / 'train'
test_dir  = DATA_DIR / 'test'

if not train_csv.exists():
    raise FileNotFoundError(f'{train_csv} introuvable. Mets train.csv dans data/.')
if not train_dir.exists() or not test_dir.exists():
    raise FileNotFoundError('Dossiers data/train et data/test attendus.')

df = read_train_csv(train_csv)
df['path'] = df['id'].map(lambda x: resolve_image_path(train_dir, x))

test_paths = sorted(test_dir.glob('*.*'), key=lambda p: int(p.stem))
test_df = pd.DataFrame({'id': [int(p.stem) for p in test_paths], 'path': test_paths})

print(df.head())
print('\nLabel distribution:')
print(df['label'].value_counts().sort_index())
print('\nTrain images:', len(df))
print('Test images :', len(test_df))
print('Example size:', Image.open(df.iloc[0]['path']).mode, Image.open(df.iloc[0]['path']).size)


   id  label                                               path
0   1      4  /home/onyxia/work/tilda-texture-classification...
1   2      6  /home/onyxia/work/tilda-texture-classification...
2   3      7  /home/onyxia/work/tilda-texture-classification...
3   4      6  /home/onyxia/work/tilda-texture-classification...
4   5      7  /home/onyxia/work/tilda-texture-classification...

Label distribution:
label
0    300
1    300
2    262
3    300
4    300
5    299
6    300
7    300
Name: count, dtype: int64

Train images: 2361
Test images : 789
Example size: L (768, 512)


## 4. Datasets et augmentations

Augmentations volontairement legeres : les classes TILDA incluent deja illumination et affine distortion, donc on evite de fabriquer des labels ambigus.


In [4]:
full_train_tfms = transforms.Compose([
    transforms.Resize(FULL_RESIZE),
    transforms.RandomCrop(FULL_IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.08, scale=(0.01, 0.04), ratio=(0.3, 3.3)),
])

full_eval_tfms = transforms.Compose([
    transforms.Resize(FULL_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

patch_train_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.RandomCrop((PATCH_SIZE, PATCH_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.05, scale=(0.01, 0.03), ratio=(0.3, 3.3)),
])

patch_center_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.CenterCrop((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

patch_eval_full_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

class TildaDataset(Dataset):
    def __init__(self, frame, transform=None, has_labels=True):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('L')
        if self.transform is not None:
            image = self.transform(image)
        label = int(row['label']) if self.has_labels else -1
        image_id = int(row['id'])
        return image, label, image_id


def make_loader(frame, transform, batch_size, shuffle=False, has_labels=True):
    ds = TildaDataset(frame, transform=transform, has_labels=has_labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=torch.cuda.is_available())


## 5. Modele SE-ResNet18 from-scratch


In [5]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


class SEBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.se = SEBlock(planes, reduction=reduction)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


class SEResNet(nn.Module):
    def __init__(self, block, layers, num_classes=8, in_channels=1):
        super().__init__()
        self.in_planes = 64
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(block, 64,  layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        self.apply(self._init_weights)

    def _make_layer(self, block, planes, blocks, stride):
        strides = [stride] + [1] * (blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0, 0.01)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


def seresnet18(num_classes=8):
    return SEResNet(SEBasicBlock, [2, 2, 2, 2], num_classes=num_classes, in_channels=1)

model = seresnet18(NUM_CLASSES)
params = sum(p.numel() for p in model.parameters())
print(f'SE-ResNet18 params: {params/1e6:.2f}M')
del model


SE-ResNet18 params: 11.26M


## 6. Fonctions train / validation / TTA


In [6]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        n += labels.size(0)
    return correct / n, total_loss / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    y_true, y_pred = [], []
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        n += labels.size(0)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
    return correct / n, total_loss / n, np.array(y_true), np.array(y_pred)


@torch.no_grad()
def predict_full_tta(model, loader):
    model.eval()
    all_probs, all_ids = [], []
    for images, _, image_ids in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        variants = [images, torch.flip(images, dims=[-1]), torch.flip(images, dims=[-2]), torch.flip(images, dims=[-2, -1])]
        probs = torch.stack([torch.softmax(model(v), dim=1) for v in variants]).mean(0)
        all_probs.append(probs.cpu())
        all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs).numpy(), np.array(all_ids)


def crop_grid(images, crop_size=384):
    _, _, h, w = images.shape
    top = [0, max((h - crop_size) // 2, 0), max(h - crop_size, 0)]
    left = [0, max((w - crop_size) // 2, 0), max(w - crop_size, 0)]
    coords = [(t, l) for t in top for l in left]
    crops = []
    for t, l in coords:
        crops.append(images[:, :, t:t+crop_size, l:l+crop_size])
    return crops


@torch.no_grad()
def predict_patch_multicrop_tta(model, loader, crop_size=384):
    model.eval()
    all_probs, all_ids = [], []
    for images, _, image_ids in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        prob_list = []
        for crop in crop_grid(images, crop_size=crop_size):
            variants = [crop, torch.flip(crop, dims=[-1]), torch.flip(crop, dims=[-2])]
            prob_list.extend([torch.softmax(model(v), dim=1) for v in variants])
        probs = torch.stack(prob_list).mean(0)
        all_probs.append(probs.cpu())
        all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs).numpy(), np.array(all_ids)


def save_submission(ids, probs, path):
    sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1).astype(int)}).sort_values('id')
    sub.to_csv(path, index=False)
    print('Saved:', path)
    print(sub['label'].value_counts().sort_index())
    return sub


## 7. Boucle 5-fold generique

Chaque fold sauvegarde : checkpoint, historique, probabilites test `.npy`, ids `.npy`, CSV fold. Si un run coupe, changer `START_FOLD_FULL` ou `START_FOLD_PATCH`.


In [7]:
def fit_fold(model, train_loader, val_loader, run_name, epochs, lr):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM,
                                weight_decay=WEIGHT_DECAY, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)

    best_acc = -1.0
    best_epoch = 0
    best_path = CHECKPOINT_DIR / f'best_{run_name}.pt'
    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_acc, train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_acc, val_loss, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        row = {
            'epoch': epoch,
            'train_acc': train_acc,
            'train_loss': train_loss,
            'val_acc': val_acc,
            'val_loss': val_loss,
            'lr': scheduler.get_last_lr()[0],
        }
        history.append(row)

        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_acc': best_acc,
                'config': {
                    'label_smoothing': LABEL_SMOOTHING,
                    'optimizer': 'SGD(momentum,nesterov)',
                    'weight_decay': WEIGHT_DECAY,
                },
            }, best_path)

        print(f'{run_name} | ep {epoch:03d} | train {train_acc:.4f}/{train_loss:.4f} | '
              f'val {val_acc:.4f}/{val_loss:.4f} | best {best_acc:.4f} @ {best_epoch}')

        if epoch - best_epoch >= PATIENCE:
            print(f'Early stop: no improvement for {PATIENCE} epochs')
            break

    elapsed = (time.time() - start_time) / 60
    hist_df = pd.DataFrame(history)
    return hist_df, best_path, best_acc, best_epoch, elapsed


def run_5fold_experiment(experiment_name, train_tfms, val_tfms, test_tfms, batch_size, epochs, lr,
                         start_fold=1, predict_kind='full'):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_probs = []
    ids_reference = None
    fold_results = []

    test_loader = make_loader(test_df, test_tfms, batch_size=batch_size, shuffle=False, has_labels=False)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['label']), start=1):
        fold_name = f'{experiment_name}_fold{fold}'
        probs_path = CHECKPOINT_DIR / f'probs_{fold_name}.npy'
        ids_path = CHECKPOINT_DIR / f'ids_{fold_name}.npy'

        if fold < start_fold:
            if probs_path.exists() and ids_path.exists():
                probs = np.load(probs_path)
                ids = np.load(ids_path)
                fold_probs.append(probs)
                ids_reference = ids if ids_reference is None else ids_reference
                print(f'Fold {fold}: loaded saved probabilities {probs.shape}')
            else:
                print(f'Fold {fold}: skipped but no saved probabilities found')
            continue

        print('\n' + '=' * 80)
        print(f'{experiment_name} | fold {fold}/{N_SPLITS}')
        print('=' * 80)

        tr_df = df.iloc[tr_idx].reset_index(drop=True)
        va_df = df.iloc[va_idx].reset_index(drop=True)
        train_loader = make_loader(tr_df, train_tfms, batch_size=batch_size, shuffle=True, has_labels=True)
        val_loader = make_loader(va_df, val_tfms, batch_size=batch_size, shuffle=False, has_labels=True)

        model = seresnet18(NUM_CLASSES).to(DEVICE)
        hist_df, best_path, best_acc, best_epoch, elapsed = fit_fold(model, train_loader, val_loader,
                                                                     fold_name, epochs, lr)
        hist_df.to_csv(OUTPUT_DIR / f'history_{fold_name}.csv', index=False)

        ckpt = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])

        if predict_kind == 'full':
            probs, ids = predict_full_tta(model, test_loader)
        elif predict_kind == 'patch':
            probs, ids = predict_patch_multicrop_tta(model, test_loader, crop_size=PATCH_SIZE)
        else:
            raise ValueError(predict_kind)

        np.save(probs_path, probs)
        np.save(ids_path, ids)
        if ids_reference is None:
            ids_reference = ids
        else:
            assert np.array_equal(ids_reference, ids), 'Test ids differ between folds'

        fold_probs.append(probs)
        save_submission(ids, probs, SUBMISSION_DIR / f'submission_{fold_name}_tta_labels0.csv')

        fold_results.append({
            'experiment': experiment_name,
            'fold': fold,
            'best_val_acc': best_acc,
            'best_epoch': best_epoch,
            'training_time_min': elapsed,
            'checkpoint': str(best_path),
        })

        del model
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(fold_results)
    results_df.to_csv(OUTPUT_DIR / f'results_{experiment_name}.csv', index=False)

    if len(fold_probs) == N_SPLITS:
        mean_probs = np.mean(fold_probs, axis=0)
        final_path = SUBMISSION_DIR / f'submission_{experiment_name}_5fold_tta_labels0.csv'
        save_submission(ids_reference, mean_probs, final_path)
        np.save(CHECKPOINT_DIR / f'probs_{experiment_name}_5fold.npy', mean_probs)
        np.save(CHECKPOINT_DIR / f'ids_{experiment_name}_5fold.npy', ids_reference)
    else:
        print(f'Pas encore de CSV 5-fold final: {len(fold_probs)}/{N_SPLITS} folds disponibles.')

    display(results_df)
    if not results_df.empty:
        print(f"Mean val acc: {results_df['best_val_acc'].mean():.4f} +/- {results_df['best_val_acc'].std():.4f}")
    return results_df


## 8. Run A — SE-ResNet18 full-image v2

C'est la version la plus proche du meilleur modele actuel, avec preprocessing propre et TTA.


In [ ]:
full_results = run_5fold_experiment(
    experiment_name='seresnet18_full_v2',
    train_tfms=full_train_tfms,
    val_tfms=full_eval_tfms,
    test_tfms=full_eval_tfms,
    batch_size=BATCH_SIZE_FULL,
    epochs=EPOCHS_FULL,
    lr=LR_FULL,
    start_fold=START_FOLD_FULL,
    predict_kind='full',
)



seresnet18_full_v2 | fold 1/5


  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

seresnet18_full_v2_fold1 | ep 001 | train 0.1827/2.2965 | val 0.1924/2.0271 | best 0.1924 @ 1


  0%|          | 0/59 [00:00<?, ?it/s]

...

seresnet18_full_v2_fold5 | ep 120 | train 0.9418/0.4621 | val 0.7055/1.2280 | best 0.7648 @ 104


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_seresnet18_full_v2_fold5_tta_labels0.csv
label
0     93
1     89
2     95
3     75
4     75
5     99
6     93
7    170
Name: count, dtype: int64
Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_seresnet18_full_v2_5fold_tta_labels0.csv
label
0     86
1     81
2     92
3     94
4     69
5     95
6     90
7    182
Name: count, dtype: int64


,experiment,fold,best_val_acc,best_epoch,training_time_min,checkpoint
0,seresnet18_full_v2,1,0.799154,95,39.830853,/home/onyxia/work/tilda-texture-classification...
1,seresnet18_full_v2,2,0.764831,114,41.795196,/home/onyxia/work/tilda-texture-classification...
2,seresnet18_full_v2,3,0.728814,117,42.532742,/home/onyxia/work/tilda-texture-classification...
3,seresnet18_full_v2,4,0.743644,109,42.203965,/home/onyxia/work/tilda-texture-classification...
4,seresnet18_full_v2,5,0.764831,104,42.288678,/home/onyxia/work/tilda-texture-classification...


Mean val acc: 0.7603 +/- 0.0265


## 9. Run B — SE-ResNet18 patch-based v2

Cette version force le reseau a regarder les details locaux. A l'inference, on moyenne une grille 3x3 de patches et des flips.


In [ ]:
patch_results = run_5fold_experiment(
    experiment_name='seresnet18_patch_v2',
    train_tfms=patch_train_tfms,
    val_tfms=patch_center_tfms,
    test_tfms=patch_eval_full_tfms,
    batch_size=BATCH_SIZE_PATCH,
    epochs=EPOCHS_PATCH,
    lr=LR_PATCH,
    start_fold=START_FOLD_PATCH,
    predict_kind='patch',
)



seresnet18_patch_v2 | fold 1/5


  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

seresnet18_patch_v2_fold1 | ep 001 | train 0.1727/2.4146 | val 0.2051/2.0671 | best 0.2051 @ 1


  0%|          | 0/59 [00:00<?, ?it/s]

...

seresnet18_patch_v2_fold2 | ep 015 | train 0.3616/1.6932 | val 0.3919/1.7212 | best 0.3919 @ 15


  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

seresnet18_patch_v2_fold2 | ep 016 | train 0.3711/1.6734 | val 0.3644/1.6260 | best 0.3919 @ 15


  0%|          | 0/60 [00:00<?, ?it/s]

## 10. Ensemble full + patch

A lancer apres les deux runs 5-fold complets. L'ensemble moyenne les soft probabilities, pas les labels.


In [ ]:
full_probs_path = CHECKPOINT_DIR / 'probs_seresnet18_full_v2_5fold.npy'
full_ids_path   = CHECKPOINT_DIR / 'ids_seresnet18_full_v2_5fold.npy'
patch_probs_path = CHECKPOINT_DIR / 'probs_seresnet18_patch_v2_5fold.npy'
patch_ids_path   = CHECKPOINT_DIR / 'ids_seresnet18_patch_v2_5fold.npy'

if full_probs_path.exists() and patch_probs_path.exists():
    full_probs = np.load(full_probs_path)
    full_ids = np.load(full_ids_path)
    patch_probs = np.load(patch_probs_path)
    patch_ids = np.load(patch_ids_path)
    assert np.array_equal(full_ids, patch_ids)

    # Full model garde un poids legerement plus fort car c'est le meilleur signal observe jusqu'ici.
    ensemble_probs = 0.60 * full_probs + 0.40 * patch_probs
    np.save(CHECKPOINT_DIR / 'probs_seresnet18_full_patch_v2_ensemble.npy', ensemble_probs)
    save_submission(full_ids, ensemble_probs,
                    SUBMISSION_DIR / 'submission_seresnet18_full_patch_v2_ensemble_labels0.csv')
else:
    print('Il manque encore les probabilites 5-fold full ou patch.')
    print('Full :', full_probs_path.exists())
    print('Patch:', patch_probs_path.exists())


## 11. Analyse rapide

Cette cellule compare les historiques des deux runs si disponibles.


In [ ]:
result_files = sorted(OUTPUT_DIR.glob('results_seresnet18_*_v2.csv'))
if result_files:
    summary = pd.concat([pd.read_csv(p) for p in result_files], ignore_index=True)
    display(summary)
    display(summary.groupby('experiment')['best_val_acc'].agg(['count', 'mean', 'std', 'min', 'max']))
else:
    print('Aucun fichier results_seresnet18_*_v2.csv pour l instant.')
